# Sheep-Herding: Scattered-Start Variation

Same task as `Herding_env.ipynb`, but the sheep start **scattered across the whole field**
instead of together in one corner. The dogs must first **gather** the flock and then
**drive** it to the target.

## Environment changes vs. the original `SheepDogEnv`

| Change | Why |
|---|---|
| Sheep spawn scattered over the field (`scatter_scale` controls how widely) | the new task |
| **Local** dog repulsion (`repulsion_radius=0.3`) instead of global `1/dist` | with global repulsion any dog pushes *every* sheep at max speed, so scattered sheep just get blasted into the walls — controlled herding is impossible |
| Weak sheep **cohesion** (attraction to neighbours within `cohesion_radius`) | pushed-together sheep clump and stay a flock |
| **Action repeat** (`decision_repeat=10` physics substeps per action, 250 decisions/episode) | thousands of tiny decisions per episode drown PPO's exploration; ~250 is learnable |
| Observation: positions sorted canonically + a `gathered` phase flag (with hysteresis) | removes permutation noise; the gather/drive phase is 1 bit of memory the MLP policy otherwise cannot infer |
| Two-phase shaped reward (gather progress -> drive progress + driving-position term + absolute distance penalty) | without the absolute penalty, a flock parked in a corner is reward-neutral and PPO stalls there |

## Training recipe (what actually worked)

Plain PPO on the full scattered task fails (0% after millions of steps) — the winning recipe is
**curriculum learning**: sheep start in a small cluster at a random location (`scatter_scale=0.1`)
and the scatter widens by 0.05 every time the recent success rate exceeds 40%, until the full
field. Trained ~6M steps total (PPO, 8 envs, lr 3e-4, gamma 0.99, net 256x256, ent_coef 0.005).

## Results (30-episode evaluations, stochastic policy)

| Policy | scatter 0.5 | scatter 0.9 | full scatter (1.0) |
|---|---|---|---|
| PPO (curriculum, ~9M steps) | **77%** | 30% | **27%** (mean ep len 222/250) |
| Scripted shepherd baseline | — | — | 84% (mean 116 steps) |
| PPO without curriculum / action repeat | — | — | 0% |

The stochastic policy herds much better than the deterministic mean action (which
scores ~0%) — the action noise breaks positional deadlocks, so evaluate/demo with
`deterministic=False`.

Outputs: model `ppo_sheep_dog_scattered.zip`, tensorboard `./Sheepherding_scattered_tensorboard/`,
GIF `Sheep-Herding-Scattered.gif`. Full experiment log in `SCATTERED_RESULTS.md`.


In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3.common.monitor import Monitor
from IPython.display import display, clear_output
import imageio
from PIL import Image
from io import BytesIO

In [ ]:
class ScatteredSheepDogEnv(gym.Env):
    """
    Variation of SheepDogEnv where the sheep start SCATTERED across the whole
    field instead of together in one corner. The dogs must first gather the
    flock and then drive it to the target position.
    """
    def __init__(self, num_sheep=25, num_dogs=5, max_speed_dog=10, max_speed_sheep=0.5, dt=0.01,
                 world_size=1.0, target_position=np.array([0.75, 0.75]), render_mode="human",
                 scatter_margin=0.05, gather_radius=0.08, cohesion_radius=0.2, cohesion_gain=0.3,
                 repulsion_radius=0.3, decision_repeat=10):
        super().__init__()
        self.num_sheep = num_sheep
        self.num_dogs = num_dogs
        self.dt = dt
        self.world_size = world_size
        self.target_position = target_position
        self.current_step = 0
        self.max_speed_dog = max_speed_dog
        self.max_speed_sheep = max_speed_sheep
        # Each policy decision is held for decision_repeat physics substeps.
        # Without this, episodes are thousands of tiny decisions and PPO's
        # exploration never composes into herding maneuvers.
        self.decision_repeat = decision_repeat
        # Scattered start needs gather + drive, which takes longer than the
        # clustered original (sheep move at most max_speed_sheep*dt per substep)
        self.max_steps = 250
        self.render_mode = render_mode
        self.fig = None
        self.ax = None
        self.scat_sheep = None
        self.scat_dogs = None
        self.target_plot = None
        self.target_distance_threshold = 0.15

        # Scatter / gathering parameters
        self.scatter_margin = scatter_margin      # keep spawns away from the walls
        self.gather_radius = gather_radius        # flock counts as "gathered" below this radius
        self.cohesion_radius = cohesion_radius    # sheep are attracted to neighbours within this range
        self.cohesion_gain = cohesion_gain        # strength of that attraction
        self.repulsion_radius = repulsion_radius  # sheep only flee dogs within this range
        # Curriculum knob: 1.0 = sheep scattered over the whole field (the real
        # task); smaller values spawn them in a proportionally smaller square at
        # a random location, which is an easier version of the same task
        self.scatter_scale = 1.0

        # Action space: (vx, vy) per dog
        self.action_space = spaces.Box(low=-self.max_speed_dog, high=self.max_speed_dog,
                                       shape=(2 * self.num_dogs,), dtype=np.float32)

        # Observation: all sheep (x,y) + all dog (x,y) + target (x,y) + gathered flag
        obs_dim = 2 * self.num_sheep + 2 * self.num_dogs + 2 + 1
        obs_low = np.full(obs_dim, 0.0, dtype=np.float32)
        obs_high = np.full(obs_dim, max(world_size, 1.0), dtype=np.float32)
        self.observation_space = spaces.Box(low=obs_low, high=obs_high, dtype=np.float32)
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        # Sheep scattered over a scatter_scale-sized square at a random location
        # (the whole field when scatter_scale=1.0), outside the target circle so
        # the episode never starts partially solved
        lo = self.scatter_margin
        hi = self.world_size - self.scatter_margin
        side = self.scatter_scale * (hi - lo)
        while True:
            ox = np.random.uniform(lo, hi - side)
            oy = np.random.uniform(lo, hi - side)
            # the square must not sit entirely inside the target circle, or the
            # out-of-circle resampling below could never terminate
            corners = np.array([[ox, oy], [ox + side, oy], [ox, oy + side], [ox + side, oy + side]])
            if np.any(np.linalg.norm(corners - self.target_position, axis=1) >= self.target_distance_threshold):
                break
        self.sheep_positions = np.stack([np.random.uniform(ox, ox + side, self.num_sheep),
                                         np.random.uniform(oy, oy + side, self.num_sheep)], axis=1)
        for i in range(self.num_sheep):
            while np.linalg.norm(self.sheep_positions[i] - self.target_position) < self.target_distance_threshold:
                self.sheep_positions[i] = np.array([np.random.uniform(ox, ox + side),
                                                    np.random.uniform(oy, oy + side)])

        # Dogs still start together in a corner
        self.dog_positions = np.random.uniform(0.0, 0.1, size=(self.num_dogs, 2))

        self._canonicalize()
        self.current_step = 0
        self.gathered = self._flock_radius() < self.gather_radius
        self.prev_mean_dist = self._mean_dist_to_target()
        self.prev_flock_radius = self._flock_radius()
        self.prev_mean_dog_dist = self._mean_dog_dist_to_flock()
        return self._get_observation(), {}

    def _canonicalize(self):
        """
        Sheep and dogs are interchangeable agents, so keep both arrays in a
        canonical (lexicographic) order. This removes permutation noise from the
        observation, which an MLP policy otherwise has to learn to ignore.
        Observation slot k and action slot k stay consistent within a step.
        """
        self.sheep_positions = self.sheep_positions[
            np.lexsort((self.sheep_positions[:, 1], self.sheep_positions[:, 0]))]
        self.dog_positions = self.dog_positions[
            np.lexsort((self.dog_positions[:, 1], self.dog_positions[:, 0]))]

    def _update_gathered(self):
        # Hysteresis: flip to "gathered" below gather_radius, back to "collecting"
        # only above 1.5x, so the phase doesn't flip-flop right at the threshold.
        r = self._flock_radius()
        if r < self.gather_radius:
            self.gathered = True
        elif r > 1.5 * self.gather_radius:
            self.gathered = False

    def _get_observation(self):
        return np.concatenate([self.sheep_positions.flatten(),
                               self.dog_positions.flatten(),
                               self.target_position.flatten(),
                               [float(self.gathered)]]).astype(np.float32)

    def _mean_dist_to_target(self):
        return np.mean(np.linalg.norm(self.sheep_positions - self.target_position, axis=1))

    def _flock_radius(self):
        center = self.sheep_positions.mean(axis=0)
        return np.mean(np.linalg.norm(self.sheep_positions - center, axis=1))

    def _mean_dog_dist_to_flock(self):
        center = self.sheep_positions.mean(axis=0)
        return np.mean(np.linalg.norm(self.dog_positions - center, axis=1))

    def step(self, action):
        action = np.clip(action, -self.max_speed_dog, self.max_speed_dog)
        dog_velocities = action.reshape(self.num_dogs, 2)
        self.current_step += 1
        for _ in range(self.decision_repeat):
            self._substep(dog_velocities)
            if self._all_in_target():
                break
        self._canonicalize()
        self._update_gathered()
        reward = self._compute_reward()
        done, truncated = self._check_done()
        info = {"is_success": done} if (done or truncated) else {}
        return self._get_observation(), reward, done, truncated, info

    def _all_in_target(self):
        d = np.linalg.norm(self.sheep_positions - self.target_position, axis=1)
        return bool(np.all(d < self.target_distance_threshold))

    def _substep(self, dog_velocities):
        self.dog_positions += dog_velocities * self.dt
        self.dog_positions = np.clip(self.dog_positions, 0.0, self.world_size)

        # ----- Sheep dynamics -----
        # 1) Repulsion from every dog (same rule as the original env, vectorised):
        #    v_i += sum_j (sheep_i - dog_j) / |sheep_i - dog_j|^2
        diff = self.sheep_positions[:, None, :] - self.dog_positions[None, :, :]
        dist_sq = np.sum(diff ** 2, axis=2, keepdims=True) + 1e-6
        # LOCAL repulsion: sheep only react to dogs within repulsion_radius.
        # With the original global 1/dist repulsion, any dog anywhere pushes every
        # sheep at max speed, so scattered sheep just get blasted into the walls
        # and controlled herding is impossible.
        in_range = dist_sq < self.repulsion_radius ** 2
        sheep_velocities = np.sum((diff / dist_sq) * in_range, axis=1)

        # 2) Weak cohesion: each sheep drifts toward the centroid of its
        #    neighbours within cohesion_radius, so pushed-together sheep clump
        #    and stay a flock once gathered
        pair_diff = self.sheep_positions[None, :, :] - self.sheep_positions[:, None, :]
        pair_dist = np.linalg.norm(pair_diff, axis=2)
        neighbor_mask = (pair_dist < self.cohesion_radius) & (pair_dist > 0)
        counts = neighbor_mask.sum(axis=1, keepdims=True)
        cohesion = (pair_diff * neighbor_mask[:, :, None]).sum(axis=1) / np.maximum(counts, 1)
        sheep_velocities += self.cohesion_gain * cohesion

        sheep_velocities = np.clip(sheep_velocities, -self.max_speed_sheep, self.max_speed_sheep)
        self.sheep_positions += sheep_velocities * self.dt
        self.sheep_positions = np.clip(self.sheep_positions, 0.0, self.world_size)

    def set_scatter_scale(self, scale):
        self.scatter_scale = float(np.clip(scale, 0.05, 1.0))

    def _compute_reward(self):
        """
        Two-phase shaping:
          - GATHER: shrinking the flock radius is rewarded strongly while the
            sheep are still spread out.
          - DRIVE: once the flock is gathered (radius < gather_radius), moving
            the flock toward the target dominates the reward.
        Dogs also get credit for approaching the flock center (not the target),
        since with scattered sheep they must reach the flock first.
        """
        distances_to_target = np.linalg.norm(self.sheep_positions - self.target_position, axis=1)
        mean_dist_to_target = distances_to_target.mean()
        flock_radius = self._flock_radius()
        mean_dog_dist_to_flock = self._mean_dog_dist_to_flock()

        gather_progress = self.prev_flock_radius - flock_radius
        drive_progress = self.prev_mean_dist - mean_dist_to_target
        dog_progress = self.prev_mean_dog_dist - mean_dog_dist_to_flock
        self.prev_flock_radius = flock_radius
        self.prev_mean_dist = mean_dist_to_target
        self.prev_mean_dog_dist = mean_dog_dist_to_flock

        gathered = self.gathered
        drive_weight = 10.0 if gathered else 2.0

        reward = 5.0 * gather_progress + drive_weight * drive_progress
        # Absolute distance penalty: without it, a gathered flock parked far from
        # the target (e.g. pinned in a corner) is reward-neutral and PPO stalls there
        reward += -1.0 * mean_dist_to_target
        reward += -0.5 * flock_radius - 0.01

        if gathered:
            # Driving-position shaping (Strombom-style): reward dogs for standing
            # just beyond the flock on the side OPPOSITE the target, where their
            # repulsion naturally pushes the flock toward the target
            center = self.sheep_positions.mean(axis=0)
            away = center - self.target_position
            away = away / (np.linalg.norm(away) + 1e-8)
            max_radius = np.max(np.linalg.norm(self.sheep_positions - center, axis=1))
            drive_point = np.clip(center + (max_radius + 0.08) * away, 0.0, self.world_size)
            mean_dog_dist_to_dp = np.mean(np.linalg.norm(self.dog_positions - drive_point, axis=1))
            reward += -2.0 * mean_dog_dist_to_dp
        else:
            reward += 1.0 * dog_progress

        # Small per-sheep in-target bonus (kept << success reward so holding the
        # flock at the target without finishing never beats terminating)
        reward += 0.2 * np.sum(distances_to_target < self.target_distance_threshold)

        if np.all(distances_to_target < self.target_distance_threshold):
            reward = 10000
        return reward

    def _check_done(self):
        done = False
        truncated = False
        distances_to_target = np.linalg.norm(self.sheep_positions - self.target_position, axis=1)
        if np.all(distances_to_target < self.target_distance_threshold):
            done = True
        elif self.current_step >= self.max_steps:
            truncated = True
        return done, truncated

    def render(self, mode="human"):
        if self.fig is None or self.ax is None:
            plt.ion()
            self.fig, self.ax = plt.subplots()
            self.ax.set_aspect("equal", adjustable="box")
            self.ax.set_xlim(0, self.world_size)
            self.ax.set_ylim(0, self.world_size)
            self.ax.set_title("Scattered Sheep-Herding Environment")
            self.ax.set_facecolor('green')
            self.scat_sheep = self.ax.scatter([], [], c="white", label="Sheep")
            self.scat_dogs = self.ax.scatter([], [], c="brown", label="Dogs")
            self.target_plot = self.ax.scatter(self.target_position[0], self.target_position[1],
                                               c="red", marker="X", s=100, label="Target")
            self.target_circle = plt.Circle(self.target_position, self.target_distance_threshold,
                                            color='red', fill=False, linestyle='--')
            self.ax.add_patch(self.target_circle)
            self.ax.legend(loc="upper left")

        self.scat_sheep.set_offsets(self.sheep_positions)
        self.scat_dogs.set_offsets(self.dog_positions)
        if self.render_mode == "human":
            clear_output(wait=True)
            display(self.fig)
        self.fig.canvas.draw()
        self.fig.canvas.flush_events()
        buf = BytesIO()
        self.fig.savefig(buf, format="png")
        buf.seek(0)
        image = Image.open(buf)
        return np.array(image)

    def close(self):
        if self.fig:
            plt.close(self.fig)
            self.fig = None
            self.ax = None

In [ ]:
# Scripted Strombom-style shepherd: collect the farthest strays, then drive the
# flock from behind. Used as a solvability check and performance baseline.
def _go(dog, goal, center, flock_rmax, max_speed):
    """Velocity toward goal, detouring AROUND the flock instead of through it."""
    v = goal - dog
    dist_goal = np.linalg.norm(v)
    # does the straight path pass close to the flock center?
    to_c = center - dog
    seg = np.linalg.norm(v) + 1e-8
    t = np.clip(np.dot(to_c, v) / seg**2, 0.0, 1.0)
    closest = dog + t * v
    clearance = flock_rmax + 0.06
    if np.linalg.norm(closest - center) < clearance and np.linalg.norm(to_c) < seg:
        # detour: steer tangentially around the flock
        away = dog - center
        away_n = np.linalg.norm(away) + 1e-8
        tang = np.array([-away[1], away[0]]) / away_n
        if np.dot(tang, v) < 0:
            tang = -tang
        v = tang + 0.5 * (away / away_n) * max(0.0, (clearance - away_n))
        dist_goal = 1.0
    out = v / (np.linalg.norm(v) + 1e-8) * max_speed
    if dist_goal < 0.05:  # damp near the goal so the dog holds position
        out *= dist_goal / 0.05
    return out


def heuristic_action(env, state=None):
    sheep, dogs = env.sheep_positions, env.dog_positions
    center = sheep.mean(axis=0)
    d_center = np.linalg.norm(sheep - center, axis=1)
    rmax = d_center.max()
    # the env tracks the gather/drive phase (with hysteresis) and exposes it in
    # the observation, so the expert is Markovian w.r.t. what the policy sees
    gathered = env.gathered
    # build one goal per dog, then match goals to dogs geometrically (nearest
    # free dog) so the plan is stable even though array order changes per step
    goals = []
    if not gathered:
        # collect the k farthest strays: stand behind each stray (opposite side
        # from flock center) and push it inward
        order = np.argsort(-d_center)
        for k in range(env.num_dogs):
            stray = sheep[order[k % len(order)]]
            away = stray - center
            n = np.linalg.norm(away) + 1e-8
            goals.append(np.clip(stray + (away / n) * 0.12, 0.0, env.world_size))
    else:
        # drive: an arc of positions behind the flock relative to the target
        away = center - env.target_position
        away = away / (np.linalg.norm(away) + 1e-8)
        base_ang = np.arctan2(away[1], away[0])
        for k in range(env.num_dogs):
            ang = base_ang + (k - (env.num_dogs - 1) / 2) * 0.4
            goal = center + (rmax + 0.10) * np.array([np.cos(ang), np.sin(ang)])
            goals.append(np.clip(goal, 0.0, env.world_size))

    action = np.zeros((env.num_dogs, 2))
    free_dogs = list(range(env.num_dogs))
    free_goals = list(range(len(goals)))
    while free_goals:
        pairs = [(np.linalg.norm(dogs[i] - goals[j]), i, j) for i in free_dogs for j in free_goals]
        _, i, j = min(pairs)
        action[i] = _go(dogs[i], goals[j], center, rmax, env.max_speed_dog)
        free_dogs.remove(i)
        free_goals.remove(j)
    return action.flatten()


# Baseline check (10 random episodes)
if False:  # set True to run the baseline probe
    succ, lens = [], []
    for seed in range(10):
        env = ScatteredSheepDogEnv(render_mode="rgb_array")
        obs, _ = env.reset(seed=seed)
        done = trunc = False
        steps = 0
        while not (done or trunc):
            obs, r, done, trunc, _ = env.step(heuristic_action(env))
            steps += 1
        succ.append(done); lens.append(steps)
    print("scripted baseline success:", np.mean(succ))

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback


class CurriculumCallback(BaseCallback):
    """Widen the sheep scatter as the policy's success rate climbs."""
    def __init__(self, start=0.15, step=0.05, threshold=0.35, min_episodes=60,
                 cooldown_steps=30_000):
        super().__init__()
        self.scale = start
        self.step_size = step
        self.threshold = threshold
        self.min_episodes = min_episodes
        # Cooldown after a promotion: episodes already in flight at the easier
        # scale still land in the stats buffer, so promoting again immediately
        # cascades on stale evidence.
        self.cooldown_steps = cooldown_steps
        self.last_promotion_step = 0

    def _on_training_start(self):
        self.training_env.env_method("set_scatter_scale", self.scale)

    def _on_rollout_end(self):
        buf = self.model.ep_info_buffer
        if (self.scale >= 1.0 or buf is None or len(buf) < self.min_episodes
                or self.num_timesteps - self.last_promotion_step < self.cooldown_steps):
            return
        succ = np.mean([e.get("is_success", False) for e in buf])
        if succ > self.threshold:
            self.scale = min(1.0, self.scale + self.step_size)
            self.training_env.env_method("set_scatter_scale", self.scale)
            buf.clear()
            self.last_promotion_step = self.num_timesteps
            print(f"[curriculum] success {succ:.2f} -> scatter_scale {self.scale:.2f} "
                  f"at {self.num_timesteps} steps", flush=True)

    def _on_step(self):
        return True


def main():
    def make_env(seed):
        def _init():
            env = ScatteredSheepDogEnv(render_mode="rgb_array")
            env.reset(seed=seed)
            return Monitor(env, info_keywords=("is_success",))
        return _init

    venv = DummyVecEnv([make_env(i) for i in range(8)])
    model = PPO("MlpPolicy", venv, verbose=1,
                tensorboard_log="./Sheepherding_scattered_tensorboard/",
                learning_rate=3e-4, clip_range=0.2, n_steps=256, batch_size=512,
                gamma=0.99, ent_coef=0.005, policy_kwargs=dict(net_arch=[256, 256]), seed=0)

    # Curriculum: sheep start in a small random cluster and the scatter widens
    # to the full field as the success rate climbs (takes several million steps)
    model.learn(total_timesteps=6_000_000, callback=CurriculumCallback(start=0.1))
    model.save("ppo_sheep_dog_scattered")

if __name__ == "__main__":
    main()

In [ ]:
test_env = ScatteredSheepDogEnv(render_mode="rgb_array")
model = PPO.load("ppo_sheep_dog_scattered")
obs, info = test_env.reset()
done = False
truncated = False
frames = []

while not done and not truncated:
    # stochastic actions herd noticeably better than deterministic ones here
    action, _ = model.predict(obs, deterministic=False)
    obs, reward, done, truncated, info = test_env.step(action)
    frames.append(test_env.render())
test_env.close()
print("success:", done, "| steps:", test_env.current_step)
imageio.mimsave("Sheep-Herding-Scattered.gif", frames, fps=30)